In [ ]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
!pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1500

regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
amount = (prices * quantities).astype(float)

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(35, 9, n).round().astype(int),
    "region": regions,
    "membership": membership,
    "channel": channels,
    "category": categories,
    "price": prices.astype(float),
    "quantity": quantities,
    "amount": amount,
})

# 오염 심기: 결측·이상치·표기 혼재
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
orders.loc[8, "customer_age"] = -3              # 불가능한 음수
orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
orders.loc[21, "region"] = "Seoul"              # 영문 표기
orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()

In [ ]:
print("응")